# Food festival

In [ ]:
import Pkg
Pkg.add("HiGHS")
Pkg.add("JuMP")


In [ ]:
using JuMP, HiGHS

include("Data/FoodFestival_data.jl")
println(Conflict) # Binary variable
println(Shifts)
println(S)
println(ConflictingShifts)


########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########
@variable(model, x[1:S, 1:S], Bin) # Schedule
@variable(model, y[1:S], Bin) # Workers

########## ---------- Objective ---------- ##########
@objective(model, Min, sum(y[i] for i in 1:S))

########## ---------- Constraint ---------- ##########
# Define y_i as a person working a shift
@constraint(model, [i in 1:S, j in 1:S], x[i,j] <= y[i])

# All rows in x must be 1
@constraint(model, [j in 1:S], sum(x[i,j] for i in 1:S) == 1 )

# If a worker works a shift with conflict, the sum of the shift + it's conflictis should be 1
@constraint(model, [i in 1:S, s in 1:S, t in 1:S; Conflict[s,t] == 1],
    x[i,s] + x[i,t] <= 1
)

########## ---------- Result ---------- ##########
optimize!(model)
println("Optimal solution:")
println(objective_value(model))
println(value.(y))

for i in 1:S
    println(value.(x[i, :]))
end

Int8[0 1 0 1 0 1 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 1 0 0 1; 1 0 1 0 0 0 1 0 1 0 0 0 0 1 0 1 1 0 0 0 0 0 1 0 0; 0 1 0 1 0 0 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 1 1 0 0; 1 0 1 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 1 1 0 0 1 1; 0 0 0 0 0 1 0 0 0 1 1 0 1 0 1 1 0 0 0 0 1 1 0 0 0; 1 0 0 0 1 0 0 1 0 0 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0; 0 1 0 0 0 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 1 1 0 0 1; 0 0 0 1 0 1 0 0 1 1 1 0 0 0 1 1 0 0 0 0 0 0 0 1 0; 0 1 1 1 0 0 1 1 0 0 0 1 1 0 1 0 0 0 1 0 1 0 0 1 0; 0 0 1 1 1 0 1 1 0 0 1 0 0 1 1 1 0 0 1 0 0 1 0 0 0; 1 0 1 0 1 1 0 1 0 1 0 1 1 0 1 0 0 0 1 1 1 0 0 0 1; 0 0 0 0 0 1 0 0 1 0 1 0 0 1 1 1 1 1 0 0 0 0 1 0 0; 0 0 0 0 1 1 0 0 1 0 1 0 0 1 0 1 0 0 1 0 0 0 1 0 1; 0 1 0 0 0 1 1 0 0 1 0 1 1 0 1 0 1 1 0 0 0 1 0 0 0; 0 0 0 0 1 1 1 1 1 1 1 1 0 1 0 1 1 0 0 0 0 0 1 1 1; 0 1 0 0 1 1 0 1 0 1 0 1 1 0 1 0 0 1 1 1 1 0 0 1 0; 1 1 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 1 0; 0 0 1 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0 0 1 1 0 0 0 0; 0 0 1 1 0 1 0 0 1 1 1 0 1 0 0 1 1 0 0 0 0 0 1 1 1; 0 0 0 1 0 0 0 0 0 0 1 0 0 